# 节点 06 — 动手实现 LSTM 和 GRU

在 README 里我们学了 LSTM 和 GRU 的**概念**。这个 notebook 的目标是：

1. **从零写出** LSTM 和 GRU（只用 NumPy）
2. **用它预测 sin 波**——验证记忆真的起作用
3. **和 PyTorch 内置版本对比**——看结果是否一致

---

**前置知识**：会 Python 列表和循环，懂 `import numpy as np`，看过 README 里的闸门公式就够了。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 非交互后端，nbconvert 可用
import matplotlib.pyplot as plt

np.random.seed(42)
print('NumPy', np.__version__, '准备好了')

## 第一步：两个最基础的函数

LSTM 和 GRU 都会用到 **sigmoid** 和 **tanh**：

- `sigmoid(x)` → 把任意数压到 0~1 之间（想象一个'阀门开度'）
- `tanh(x)` → 把任意数压到 -1~1 之间（想象一个'信息强度'）

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

assert abs(sigmoid(0) - 0.5) < 1e-9
assert sigmoid(1000) > 0.999
assert sigmoid(-1000) < 0.001
print('sigmoid 通过')

assert abs(np.tanh(0)) < 1e-9
assert np.tanh(100) > 0.999
print('tanh   通过')

## 第二步：从零实现 LSTM

回忆 README 里的三道闸门：

| 闸门 | 作用 |
|------|------|
| 遗忘闸 | 决定忘掉多少旧记忆 |
| 输入闸 | 决定写入多少新信息 |
| 输出闸 | 决定输出多少给下一层 |

细胞更新：$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$

隐藏状态：$h_t = o_t \odot \tanh(c_t)$

In [ ]:
class LSTMNumpy:
    """从零实现的单层 LSTM，只用 NumPy。"""

    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        d = input_dim + hidden_dim
        scale = 0.1
        self.Wf = np.random.randn(hidden_dim, d) * scale
        self.Wi = np.random.randn(hidden_dim, d) * scale
        self.Wc = np.random.randn(hidden_dim, d) * scale
        self.Wo = np.random.randn(hidden_dim, d) * scale
        self.bf = np.zeros(hidden_dim)
        self.bi = np.zeros(hidden_dim)
        self.bc = np.zeros(hidden_dim)
        self.bo = np.zeros(hidden_dim)
        self.Wy = np.random.randn(1, hidden_dim) * scale
        self.by = np.zeros(1)

    def step(self, x_t, h_prev, c_prev):
        combined = np.concatenate([h_prev, x_t])
        f = sigmoid(self.Wf @ combined + self.bf)
        i = sigmoid(self.Wi @ combined + self.bi)
        c_cand = np.tanh(self.Wc @ combined + self.bc)
        o = sigmoid(self.Wo @ combined + self.bo)
        c_new = f * c_prev + i * c_cand
        h_new = o * np.tanh(c_new)
        return h_new, c_new

    def forward(self, sequence):
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        outputs = []
        for x_t in sequence:
            x_t = np.array([x_t])
            h, c = self.step(x_t, h, c)
            y = self.Wy @ h + self.by
            outputs.append(float(y))
        return outputs

lstm = LSTMNumpy(input_dim=1, hidden_dim=4)
dummy_seq = [0.1, 0.2, 0.3, 0.4, 0.5]
out = lstm.forward(dummy_seq)
assert len(out) == 5
assert all(isinstance(v, float) for v in out)
print('LSTM 结构验证通过')
print('示例输出（未训练）:', [f'{v:.4f}' for v in out])

## 第三步：从零实现 GRU

GRU 只有**两道闸**：更新闸 $z_t$ 和重置闸 $r_t$。

| 闸门 | 作用 |
|------|------|
| 更新闸 $z_t$ | 控制保留多少旧状态 vs 写入多少新内容 |
| 重置闸 $r_t$ | 控制计算新内容时看多少旧状态 |

更新：$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

注意：$z_t \approx 0$ 时几乎保留旧状态（记住），$z_t \approx 1$ 时几乎全用新内容（更新）。

In [ ]:
class GRUNumpy:
    """从零实现的单层 GRU，只用 NumPy。"""

    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        d = input_dim + hidden_dim
        scale = 0.1
        self.Wz = np.random.randn(hidden_dim, d) * scale
        self.Wr = np.random.randn(hidden_dim, d) * scale
        self.Wh = np.random.randn(hidden_dim, d) * scale
        self.bz = np.zeros(hidden_dim)
        self.br = np.zeros(hidden_dim)
        self.bh = np.zeros(hidden_dim)
        self.Wy = np.random.randn(1, hidden_dim) * scale
        self.by = np.zeros(1)

    def step(self, x_t, h_prev):
        combined = np.concatenate([h_prev, x_t])
        z = sigmoid(self.Wz @ combined + self.bz)
        r = sigmoid(self.Wr @ combined + self.br)
        combined_r = np.concatenate([r * h_prev, x_t])
        h_cand = np.tanh(self.Wh @ combined_r + self.bh)
        h_new = (1 - z) * h_prev + z * h_cand
        return h_new

    def forward(self, sequence):
        h = np.zeros(self.hidden_dim)
        outputs = []
        for x_t in sequence:
            x_t = np.array([x_t])
            h = self.step(x_t, h)
            y = self.Wy @ h + self.by
            outputs.append(float(y))
        return outputs

gru = GRUNumpy(input_dim=1, hidden_dim=4)
out_gru = gru.forward(dummy_seq)
assert len(out_gru) == 5
print('GRU 结构验证通过')
print('示例输出（未训练）:', [f'{v:.4f}' for v in out_gru])

## 第四步：验证 LSTM 梯度可以下降

用**数值微分**（不推导反向传播）验证权重的梯度存在——
这说明 LSTM 是可以被训练的。

**任务**：给定 sin 波的前 N 个点，预测下一个点。

In [ ]:
T = 40
seq_len = 10
t_all = np.linspace(0, 4 * np.pi, T + 1)
data = np.sin(t_all)
X_train = [data[i:i+seq_len] for i in range(T - seq_len)]
y_train = [data[i+seq_len] for i in range(T - seq_len)]
print(f'训练样本数：{len(X_train)}')
print(f'第一个输入（前5步）：{X_train[0][:5].round(3)}')
print(f'对应目标值：{y_train[0]:.3f}')

def mse_loss(predictions, targets):
    errors = [(p - t) ** 2 for p, t in zip(predictions, targets)]
    return sum(errors) / len(errors)

def compute_loss(model, X_list, y_list):
    preds = [model.forward(x)[-1] for x in X_list]
    return mse_loss(preds, y_list)

lstm_test = LSTMNumpy(input_dim=1, hidden_dim=8)
loss_before = compute_loss(lstm_test, X_train, y_train)
eps = 1e-5
Wy_orig = lstm_test.Wy.copy()
lstm_test.Wy[0, 0] += eps
loss_plus = compute_loss(lstm_test, X_train, y_train)
grad_approx = (loss_plus - loss_before) / eps
lstm_test.Wy = Wy_orig
print(f'训练前 MSE：{loss_before:.4f}')
print(f'Wy[0,0] 的数值梯度：{grad_approx:.6f}')
assert grad_approx != 0.0, '梯度应非零（网络可训练）'
print('梯度非零验证通过 — LSTM 可训练')

## 第五步：和 PyTorch 内置版本对比

PyTorch 的 `nn.LSTM` 和 `nn.GRU` 是生产级实现。
我们验证：**在相同权重下，手写版和 PyTorch 的计算结果是否一致？**

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} 可用')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch 未安装，跳过对比验证')

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(0)
    torch_lstm = nn.LSTM(input_size=1, hidden_size=4, batch_first=True)
    test_seq = [0.1, 0.5, -0.3, 0.8, -0.6]
    x_tensor = torch.tensor(test_seq, dtype=torch.float32).view(1, 5, 1)
    with torch.no_grad():
        output, (h_n, c_n) = torch_lstm(x_tensor)
    print('PyTorch LSTM 最后一步 h:', h_n.squeeze().numpy().round(4))

    with torch.no_grad():
        Wih = torch_lstm.weight_ih_l0.numpy()
        Whh = torch_lstm.weight_hh_l0.numpy()
        bih = torch_lstm.bias_ih_l0.numpy()
        bhh = torch_lstm.bias_hh_l0.numpy()

    h = 4
    idx = {'i': slice(0*h, 1*h), 'f': slice(1*h, 2*h),
           'g': slice(2*h, 3*h), 'o': slice(3*h, 4*h)}

    lstm_copy = LSTMNumpy(input_dim=1, hidden_dim=4)
    lstm_copy.Wf = np.hstack([Whh[idx['f']], Wih[idx['f']]])
    lstm_copy.Wi = np.hstack([Whh[idx['i']], Wih[idx['i']]])
    lstm_copy.Wc = np.hstack([Whh[idx['g']], Wih[idx['g']]])
    lstm_copy.Wo = np.hstack([Whh[idx['o']], Wih[idx['o']]])
    lstm_copy.bf = bih[idx['f']] + bhh[idx['f']]
    lstm_copy.bi = bih[idx['i']] + bhh[idx['i']]
    lstm_copy.bc = bih[idx['g']] + bhh[idx['g']]
    lstm_copy.bo = bih[idx['o']] + bhh[idx['o']]

    h_np = np.zeros(4)
    c_np = np.zeros(4)
    for x_t in test_seq:
        h_np, c_np = lstm_copy.step(np.array([x_t]), h_np, c_np)
    print('手写 LSTM 最后一步 h:', h_np.round(4))
    diff = np.max(np.abs(h_np - h_n.squeeze().numpy()))
    print(f'最大差值：{diff:.2e}')
    assert diff < 1e-5, f'手写版和PyTorch差距过大：{diff}'
    print('手写 LSTM 与 PyTorch 结果一致（差值 < 1e-5）')
else:
    print('跳过 PyTorch LSTM 对比（未安装）')

In [ ]:
# GRU 闸门值范围验证（无论是否有 PyTorch 都可运行）
gru_test = GRUNumpy(input_dim=1, hidden_dim=4)
h_test = np.zeros(4)
x_test = np.array([0.5])
combined = np.concatenate([h_test, x_test])
z_np = sigmoid(gru_test.Wz @ combined + gru_test.bz)
r_np = sigmoid(gru_test.Wr @ combined + gru_test.br)
assert all(0 <= v <= 1 for v in z_np), '更新闸应在 0~1'
assert all(0 <= v <= 1 for v in r_np), '重置闸应在 0~1'
print('GRU 更新闸值:', z_np.round(4))
print('GRU 重置闸值:', r_np.round(4))
print('GRU 闸门值范围验证通过（所有值在 0~1 之间）')

## 第六步：可视化 sin 波预测

用随机初始化（未训练）的 LSTM 和 GRU 预测 sin 波——
预测结果与真实值相差很大，这很正常：记忆能力需要训练。

In [ ]:
lstm_vis = LSTMNumpy(input_dim=1, hidden_dim=8)
gru_vis  = GRUNumpy(input_dim=1, hidden_dim=8)

preds_lstm = [lstm_vis.forward(x)[-1] for x in X_train]
preds_gru  = [gru_vis.forward(x)[-1]  for x in X_train]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_axis = list(range(len(y_train)))
for ax, preds, name in zip(axes, [preds_lstm, preds_gru], ['LSTM（未训练）', 'GRU（未训练）']):
    ax.plot(x_axis, y_train, 'b-o', markersize=4, label='真实值 (sin)')
    ax.plot(x_axis, preds, 'r--s', markersize=4, label='模型预测')
    ax.set_title(name)
    ax.legend()
    ax.set_xlabel('时间步')
    ax.set_ylabel('sin 值')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lstm_gru_untrained.png', dpi=72, bbox_inches='tight')
plt.close()
print('图片保存为 lstm_gru_untrained.png')
loss_lstm = mse_loss(preds_lstm, y_train)
loss_gru  = mse_loss(preds_gru, y_train)
print(f'未训练 LSTM MSE：{loss_lstm:.4f}')
print(f'未训练 GRU  MSE：{loss_gru:.4f}')
print('（都很差——记忆能力需要学习）')

## 第七步：LSTM vs GRU 对比总结

| 指标 | LSTM | GRU |
|------|------|-----|
| 闸门数量 | 3（遗忘、输入、输出） | 2（更新、重置） |
| 状态变量 | h + c（两个向量） | 只有 h（一个向量） |
| 参数量（相同 hidden_dim） | 较多 | 少约 25% |
| 实践效果 | 相近 | 相近（Chung et al. 2014） |
| 适合场景 | 超长序列、精细控制 | 数据少、训练速度优先 |

**核心共同点**：两者都用「闸门」解决梯度消失——当闸门接近 1 时，梯度可以无损通过，
网络能记住很久之前的信息。

In [ ]:
print('=' * 50)
print('所有验证结果汇总')
print('=' * 50)
print('sigmoid / tanh 函数正确')
print('LSTMNumpy 结构正确（forward 输出长度=序列长度）')
print('GRUNumpy  结构正确（forward 输出长度=序列长度）')
print('梯度（数值微分）非零 — LSTM 可训练')
print('GRU 更新闸/重置闸输出值在 [0, 1] 范围内')
print('可视化图片已保存')
print()
print('节点 06 notebook 完成')